In [1]:
import os
import numpy as np
import tensorflow as tf
import psutil
import gc
import time
import pandas as pd
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler

2025-03-22 18:12:28.187722: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-22 18:12:28.199938: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742681548.214369  339435 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742681548.218708  339435 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-22 18:12:28.232804: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

# Preprocessing

In [2]:
def load_file(path):
    data = pd.read_csv(path, sep=',', low_memory=False)
    return data

In [3]:
import pandas as pd

# File paths
base_path = "/home/uottawa.o.univ/hpate033/GAN Project/CIC-DDoS2019/"
files = [
    "DrDoS_LDAP.csv", "DrDoS_MSSQL.csv", "DrDoS_NetBIOS.csv",
    "DrDoS_NTP.csv", "DrDoS_SNMP.csv", "DrDoS_SSDP.csv",
    "DrDoS_UDP.csv", "DrDoS_DNS.csv"
]

# Load first file
ddos_df = load_file(base_path + files[0])  # Expecting ONE DataFrame
print('file 1 loaded')

# Load remaining files
for i, file in enumerate(files[1:], start=2):
    df = load_file(base_path + file)  # Expecting ONE DataFrame
    
    # Concatenate the new file data
    ddos_df = pd.concat([ddos_df, df], ignore_index=True)

    print(f'file {i} loaded')

del df

file 1 loaded
file 2 loaded
file 3 loaded
file 4 loaded
file 5 loaded
file 6 loaded
file 7 loaded
file 8 loaded


In [4]:
import pandas as pd
import numpy as np
import hashlib


ddos_df.info()

# Function to convert string to numeric hash
def string2numeric_hash(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)

# Replace infinite values
ddos_df = ddos_df.replace(['Infinity', np.inf], 0)

# Convert numerical columns safely
ddos_df[' Flow Packets/s'] = pd.to_numeric(ddos_df[' Flow Packets/s'], errors='coerce').fillna(0)
ddos_df['Flow Bytes/s'] = pd.to_numeric(ddos_df['Flow Bytes/s'], errors='coerce').fillna(0)

# Convert labels to numeric
ddos_df[' Label'] = ddos_df[' Label'].replace({
    'BENIGN': 0, 'DrDoS_DNS': 1, 'DrDoS_LDAP': 1, 'DrDoS_MSSQL': 1,
    'DrDoS_NTP': 1, 'DrDoS_NetBIOS': 1, 'DrDoS_SNMP': 1, 'DrDoS_SSDP': 1,
    'DrDoS_UDP': 1
}).astype(int)

# Ensure no NaN timestamps before splitting
ddos_df[' Timestamp'] = ddos_df[' Timestamp'].fillna('1970-01-01 00:00:00.000000')

# Drop unnecessary columns
ddos_df.drop(columns=[' Timestamp', ' Source IP', ' Destination IP', 'Flow ID', 'SimillarHTTP', 'Unnamed: 0'], inplace=True)

# Remove space in columns
ddos_df.columns = ddos_df.columns.str.strip()

print('Data processed successfully!')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28001999 entries, 0 to 28001998
Data columns (total 88 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   Unnamed: 0                    int64  
 1   Flow ID                       object 
 2    Source IP                    object 
 3    Source Port                  int64  
 4    Destination IP               object 
 5    Destination Port             int64  
 6    Protocol                     int64  
 7    Timestamp                    object 
 8    Flow Duration                int64  
 9    Total Fwd Packets            int64  
 10   Total Backward Packets       int64  
 11  Total Length of Fwd Packets   float64
 12   Total Length of Bwd Packets  float64
 13   Fwd Packet Length Max        float64
 14   Fwd Packet Length Min        float64
 15   Fwd Packet Length Mean       float64
 16   Fwd Packet Length Std        float64
 17  Bwd Packet Length Max         float64
 18   Bwd Packet Length M

In [5]:
print(ddos_df.shape)

ddos_df.info()

print(ddos_df.columns)

(28001999, 82)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28001999 entries, 0 to 28001998
Data columns (total 82 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Source Port                  int64  
 1   Destination Port             int64  
 2   Protocol                     int64  
 3   Flow Duration                int64  
 4   Total Fwd Packets            int64  
 5   Total Backward Packets       int64  
 6   Total Length of Fwd Packets  float64
 7   Total Length of Bwd Packets  float64
 8   Fwd Packet Length Max        float64
 9   Fwd Packet Length Min        float64
 10  Fwd Packet Length Mean       float64
 11  Fwd Packet Length Std        float64
 12  Bwd Packet Length Max        float64
 13  Bwd Packet Length Min        float64
 14  Bwd Packet Length Mean       float64
 15  Bwd Packet Length Std        float64
 16  Flow Bytes/s                 float64
 17  Flow Packets/s               float64
 18  Flow IAT Mean            

In [6]:
# Step 1. Load and preprocess the dataset.

# Print the available columns to double-check the header.
print("Dataset columns:\n", ddos_df.columns.tolist())

# Check if the 'Label' column exists.
if 'Label' not in ddos_df.columns:
    raise KeyError("Column 'Label' not found in the dataset. Please verify the CSV header.")

# Filter the dataset to include only attack samples (assume attack samples have Label == 1)
attack_data = ddos_df[ddos_df['Label'] == 1]

# Separate features (drop the Label column)
X = attack_data.drop(columns=['Label'])
feature_names = X.columns  # Save feature names for later interpretation
X = X.values

# Normalize features to the [0, 1] range
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
del X
num_samples, num_features = X_scaled.shape
print(f"Loaded {num_samples} attack samples with {num_features} features.")

Dataset columns:
 ['Source Port', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG F

In [28]:
print(feature_names)

Index(['Source Port', 'Destination Port', 'Protocol', 'Flow Duration',
       'Total Fwd Packets', 'Total Backward Packets',
       'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag 

# GANFS Algorithn

In [7]:
# Step 0. Check for GPU and configure memory growth.
gpus = tf.config.list_physical_devices('GPU')
print(gpus)
if gpus:
    try:
        for gpu in gpus:
            # Enable memory growth so that TensorFlow does not allocate all GPU memory
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU detected and memory growth enabled.")
    except RuntimeError as e:
        print("Error setting memory growth: ", e)
else:
    print("No GPU detected; running on CPU.")

device = '/GPU:0' if gpus else '/CPU:0'
print(device)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU detected and memory growth enabled.
/GPU:0


In [8]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [9]:
# Step 2. Build the GAN models.
def build_generator(input_dim, output_dim):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_dim=input_dim),
        layers.Dense(128, activation='relu'),
        layers.Dense(output_dim, activation='sigmoid')
    ])
    return model

def build_discriminator(input_dim):
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_dim=input_dim),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

with tf.device(device):
    # Build generator and discriminator
    generator = build_generator(num_features, num_features)
    discriminator = build_discriminator(num_features)

    # Compile the discriminator
    discriminator.compile(loss='binary_crossentropy',
                          optimizer=optimizers.Adam(learning_rate=0.001),
                          metrics=['accuracy'])

    # Create the GAN model
    discriminator.trainable = True
    gan_input = layers.Input(shape=(num_features,))
    generated_sample = generator(gan_input)
    gan_output = discriminator(generated_sample)
    gan = models.Model(gan_input, gan_output)
    gan.compile(loss='binary_crossentropy', optimizer=optimizers.Adam(learning_rate=0.001))

print("Generator, Discriminator, and GAN models are built and compiled.")

I0000 00:00:1742658935.025712  485859 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8085 MB memory:  -> device: 0, name: NVIDIA H100 PCIe MIG 1g.10gb, pci bus id: 0000:65:00.0, compute capability: 9.0
/usr/lib/python3/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Generator, Discriminator, and GAN models are built and compiled.


In [10]:
discriminator.compile(loss='binary_crossentropy',
                          optimizer=optimizers.Adam(learning_rate=0.001),
                          metrics=['accuracy'])

In [11]:
print(discriminator.optimizer)  # Should print the optimizer details
print(discriminator.loss)  # Should print the loss function
print(discriminator.metrics)  # Should print the metrics

binary_crossentropy
[<Mean name=loss>, <CompileMetrics name=compile_metrics>]


In [12]:
# Ensure the checkpoint directory exists
checkpoint_dir = './training_checkpoints'
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")

# Assume these objects are created elsewhere in your code:
# generator, discriminator, gan, gen_optimizer, disc_optimizer

# Create a checkpoint instance that tracks models and optimizer states
checkpoint = tf.train.Checkpoint(generator=generator,
                                 discriminator=discriminator,
                                 gan=gan)

# Define training parameters
epochs = 500
batch_size = 4096

with tf.device(device):
    print("Starting GAN training on", device)
    for epoch in range(epochs):
        # ---------------------
        # Train the discriminator
        # ---------------------
        # Get the total number of samples and select a random batch
        num_samples = X_scaled.shape[0]
        idx = np.random.choice(num_samples, batch_size, replace=False)
        real_samples = X_scaled[idx]

        # Generate a synthetic batch from noise
        noise = np.random.normal(0, 1, (batch_size, num_features))
        fake_samples = generator.predict(noise, verbose=0)

        # Use label smoothing for more stable training
        real_labels = 0.9 * np.ones((batch_size, 1))
        fake_labels = 0.1 * np.ones((batch_size, 1))

        # Train on real and fake samples
        d_loss_real = discriminator.train_on_batch(real_samples, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_samples, fake_labels)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # ---------------------
        # Train the generator
        # ---------------------
        noise = np.random.normal(0, 1, (batch_size, num_features))
        valid_y = np.ones((batch_size, 1))  # Generator aims for discriminator to output 1
        g_loss = gan.train_on_batch(noise, valid_y)

        # Every 20 epochs, display progress and save a checkpoint
        if epoch % 20 == 0:
            # If the discriminator loss returns both loss and accuracy info (e.g. [loss, acc]),
            # extract them; otherwise, use defaults.
            d_loss_value = d_loss[0] if isinstance(d_loss, (list, np.ndarray)) else d_loss
            d_acc = d_loss[1] if isinstance(d_loss, (list, np.ndarray)) and len(d_loss) > 1 else 0.0

            # Print formatted outputs with zero-padded epoch and formatted loss/accuracy.
            print(f"Epoch {epoch:04d}: D_loss = {d_loss_value:.4f} (acc: {100*d_acc:05.2f}%), G_loss = {g_loss:.4f}")
            # Save the current training state
            checkpoint.save(file_prefix=checkpoint_prefix)

    print("GAN training finished.")


Starting GAN training on /GPU:0


I0000 00:00:1742658936.764324  488007 service.cc:148] XLA service 0x5564c2a96080 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1742658936.764348  488007 service.cc:156]   StreamExecutor device (0): NVIDIA H100 PCIe MIG 1g.10gb, Compute Capability 9.0
2025-03-22 11:55:36.767864: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1742658936.787465  488007 cuda_dnn.cc:529] Loaded cuDNN version 90501
I0000 00:00:1742658941.682979  488007 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2025-03-22 11:55:44.625924: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164', 100 bytes spill stores, 88 bytes spill loads

2025-03-22 11:55:45.737298: I externa

Epoch 0000: D_loss = 0.6728 (acc: 00.00%), G_loss = 0.7377


Epoch 0020: D_loss = 0.5547 (acc: 00.00%), G_loss = 0.8019
Epoch 0040: D_loss = 0.4991 (acc: 00.00%), G_loss = 0.9266
Epoch 0060: D_loss = 0.4659 (acc: 00.00%), G_loss = 1.0262
Epoch 0080: D_loss = 0.4455 (acc: 00.00%), G_loss = 1.0975
Epoch 0100: D_loss = 0.4323 (acc: 00.00%), G_loss = 1.1493
Epoch 0120: D_loss = 0.4211 (acc: 00.00%), G_loss = 1.2032
Epoch 0140: D_loss = 0.4150 (acc: 00.00%), G_loss = 1.2257
Epoch 0160: D_loss = 0.4114 (acc: 00.00%), G_loss = 1.2466
Epoch 0180: D_loss = 0.4068 (acc: 00.00%), G_loss = 1.2839
Epoch 0200: D_loss = 0.4030 (acc: 00.00%), G_loss = 1.3118
Epoch 0220: D_loss = 0.3984 (acc: 00.00%), G_loss = 1.3476
Epoch 0240: D_loss = 0.3952 (acc: 00.00%), G_loss = 1.3762
Epoch 0260: D_loss = 0.3913 (acc: 00.00%), G_loss = 1.4187
Epoch 0280: D_loss = 0.3878 (acc: 00.00%), G_loss = 1.4576
Epoch 0300: D_loss = 0.3846 (acc: 00.00%), G_loss = 1.4937
Epoch 0320: D_loss = 0.3819 (acc: 00.00%), G_loss = 1.5209
Epoch 0340: D_loss = 0.3794 (acc: 00.00%), G_loss = 1.54

# Calculating predictions and sensitivity

In [13]:
def compute_baseline_predictions(model, data, safety_factor=0.7):
    """Memory-optimized prediction generator using direct batch generation."""
    print("🚀 Initializing memory-safe prediction pipeline...")
    print(f"• Input shape: {data.shape}")
    print(f"• Initial RAM: {psutil.virtual_memory().available/1e9:.2f} GB\n")

    # Convert to float16 if possible to reduce memory footprint
    if model.dtype == tf.float16:
        data = data.astype(np.float16)
        print("• Using float16 precision")
    else:
        data = data.astype(np.float32)
        print("• Using float32 precision")

    def calculate_batch_size():
        bytes_per_sample = data.shape[1] * (2 if data.dtype == np.float16 else 4)
        available_mem = psutil.virtual_memory().available * safety_factor
        max_batch = int(available_mem / (bytes_per_sample * 3))
        return max(512, min(max_batch, 4194304))

    # Configure TensorFlow for optimal large dataset handling
    tf.config.optimizer.set_jit(True)

    optimal_batch_size = calculate_batch_size()
    print(f"\n⚙️ Initial batch size: {optimal_batch_size}")
    
    def data_generator():
        for i in range(0, len(data), optimal_batch_size):
            yield data[i:i+optimal_batch_size]

    baseline_preds = None
    retry_count = 0
    start_time = time.time()

    print(f"\n⚙️ Initial batch size: {optimal_batch_size}")
    print("──────────────────────────────────────────────────")

    while True:
        try:
            # Create dataset from generator with proper typing
            dataset = tf.data.Dataset.from_generator(
                data_generator,
                output_signature=tf.TensorSpec(shape=(None, data.shape[1]), dtype=data.dtype)
            ).prefetch(tf.data.AUTOTUNE)

            total_batches = np.ceil(len(data) / optimal_batch_size).astype(int)
            
            if baseline_preds is None:
                print("\n🔨 Allocating prediction array...")
                sample_pred = model(data[:1]).numpy()
                baseline_preds = np.empty((len(data), *sample_pred.shape[1:]), 
                                       dtype=sample_pred.dtype)
                print(f"• Prediction array size: {baseline_preds.nbytes/1e9:.2f} GB")
                print("──────────────────────────────────────────────────")

            print(f"\n🔍 Starting batch processing (attempt {retry_count+1})...")
            current_idx = 0
            for batch_idx, batch in enumerate(dataset):
                pred = model(batch, training=False).numpy()
                batch_size = pred.shape[0]
                baseline_preds[current_idx:current_idx + batch_size] = pred
                current_idx += batch_size

                # Memory management
                del pred, batch
                if batch_idx % 10 == 0:
                    tf.keras.backend.clear_session()
                    gc.collect()

                # Progress reporting
                if batch_idx % 100 == 0 or batch_idx == total_batches-1:
                    elapsed = time.time() - start_time
                    mem = psutil.virtual_memory()
                    progress = current_idx/len(data)*100
                    print(
                        f"\r▏{'█' * int(progress/2)}{' ' * (50 - int(progress/2))}▏ "
                        f"{progress:.1f}% • "
                        f"Batch {batch_idx+1}/{total_batches} • "
                        f"RAM: {mem.used/1e9:.1f}/{mem.total/1e9:.1f} GB • "
                        f"Elapsed: {elapsed//60:.0f}m {elapsed%60:.0f}s",
                        end="", flush=True
                    )

            break

        except tf.errors.ResourceExhaustedError:
            retry_count += 1
            print(f"\n\n⚠️ Memory overload at batch size {optimal_batch_size}!")
            optimal_batch_size = max(optimal_batch_size // 2, 128)
            print(f"🔄 Retrying with batch size: {optimal_batch_size}")
            print("──────────────────────────────────────────────────")

    # Finalization
    total_time = time.time() - start_time
    
    print(f"\n\n🎉 Successfully processed {len(data):,} samples!")
    print(f"• Final batch size: {optimal_batch_size}")
    print(f"• Total duration: {total_time//60:.0f}m {total_time%60:.0f}s")
    print(f"• Peak RAM usage: {psutil.virtual_memory().percent}%")
    print("──────────────────────────────────────────────────")

    return baseline_preds, optimal_batch_size


In [14]:
# Generate baseline predictions
baseline_preds, batch_size = compute_baseline_predictions(
    model=discriminator,
    data=X_scaled,
    safety_factor=0.7  # Adjust based on your system
)

🚀 Initializing memory-safe prediction pipeline...
• Input shape: (27974480, 81)
• Initial RAM: 453.94 GB

• Using float32 precision

⚙️ Initial batch size: 4194304

⚙️ Initial batch size: 4194304
──────────────────────────────────────────────────

🔨 Allocating prediction array...
• Prediction array size: 0.11 GB
──────────────────────────────────────────────────

🔍 Starting batch processing (attempt 1)...


2025-03-22 12:09:27.407315: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:497] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.00GiB (rounded to 2147483648)requested by op AddV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-03-22 12:09:27.407576: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1053] BFCAllocator dump for GPU_0_bfc
2025-03-22 12:09:27.407622: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (256): 	Total Chunks: 44, Chunks in use: 44. 11.0KiB allocated for chunks. 11.0KiB in use in bin. 3.4KiB client-requested in use in bin.
2025-03-22 12:09:27.407653: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (512): 	Total Chunks: 11, Chunks in use: 11. 5.5KiB allocated for chunks. 5.5KiB in use in bin. 4.9KiB client-requested in use in bin.
2025-03-22 



⚠️ Memory overload at batch size 4194304!
🔄 Retrying with batch size: 2097152
──────────────────────────────────────────────────

🔍 Starting batch processing (attempt 2)...
▏██████████████████████████████████████████████████▏ 100.0% • Batch 14/14 • RAM: 96.3/540.5 GB • Elapsed: 0m 28s

2025-03-22 12:09:43.528587: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence




🎉 Successfully processed 27,974,480 samples!
• Final batch size: 2097152
• Total duration: 0m 31s
• Peak RAM usage: 18.4%
──────────────────────────────────────────────────


In [15]:
batch_size = 2097152

In [16]:
def improved_sensitivity_analysis(model, data, baseline_preds, batch_size,
                                 perturbation_mode='dynamic',
                                 perturbation_factors=[0.5, 1.0, 2.0, 5.0],
                                 safety_factor=0.6):
    """Sensitivity analysis with neighbor-based dynamic perturbations"""
    n_samples, n_features = data.shape
    sensitivities = np.zeros(n_features, dtype=np.float32)
    start_time = time.time()

    # Precompute neighbor differences for dynamic mode
    if perturbation_mode == 'dynamic':
        def calculate_feature_granularity(feature_data):
            sorted_values = np.sort(feature_data)
            diffs = np.diff(sorted_values)
            non_zero_diffs = diffs[diffs > 0]
            if len(non_zero_diffs) == 0:
                return max(1e-3, 0.01 * (np.max(feature_data) - np.min(feature_data)))
            return np.mean(non_zero_diffs)
        
        base_deltas = np.array([calculate_feature_granularity(data[:, i]) 
                              for i in range(n_features)])
        print(base_deltas)
        print("🔍 Computed feature granularity based on neighbor differences")
    else:
        print("⚖️ Using static perturbation levels")

    total_operations = n_features * len(perturbation_factors) * 2
    print(f"\n🚀 Launching sensitivity analysis with:")
    print(f"• Features: {n_features} | Perturbations: {len(perturbation_factors)}")
    print(f"• Total operations: {total_operations:,}")
    print(f"• Batch size: {batch_size:,} samples")
    print(f"• Perturbation mode: {perturbation_mode.upper()}")
    print("──────────────────────────────────────────────────")

    for feat_idx in range(n_features):
        feature_start = time.time()
        progress_bar = f"[{'▋'*(20*(feat_idx+1)//n_features)}{' '*(20-20*(feat_idx+1)//n_features)}]"
        print(f"\n🔧 Feature {feat_idx+1}/{n_features} {progress_bar}")
        print(f"• Current RAM: {psutil.virtual_memory().percent}% used")
        print(f"• Elapsed: {time.time()-start_time:.1f}s")
        
        # Determine perturbation magnitudes
        if perturbation_mode == 'dynamic':
            base_Δ = base_deltas[feat_idx]
            deltas = [f * base_Δ for f in perturbation_factors]
            delta_info = f"(base Δ={base_Δ:.4f} → deltas: {[f'{d:.4f}' for d in deltas]})"
        else:
            deltas = perturbation_factors
            delta_info = f"(static deltas: {perturbation_factors})"
            
        print(f"• Perturbation strategy: {delta_info}")
        feature_sens = 0.0

        for delta_idx, delta in enumerate(deltas):
            delta_start = time.time()
            delta_sens = 0.0
            print(f"\n  🎚️ Perturbation {delta_idx+1}/{len(deltas)} (δ={delta:.4f})")
            
            for direction in [1, -1]:
                dir_start = time.time()
                total_diff = 0.0
                batches_processed = 0
                print(f"  ↕️ Direction: {'+' if direction >0 else '-'}")
                
                for start in range(0, n_samples, batch_size):
                    batch_data = data[start:start+batch_size].copy()
                    original_values = batch_data[:, feat_idx]
                    
                    # Apply smart perturbation with neighbor awareness
                    perturbed = original_values + direction * delta
                    batch_data[:, feat_idx] = np.clip(
                        np.where(
                            (perturbed >= np.min(original_values)) & 
                            (perturbed <= np.max(original_values)),
                            perturbed,
                            original_values  # Preserve original if beyond natural range
                        ), 0, 1
                    )
                    
                    perturbed_preds = model(batch_data).numpy()
                    total_diff += np.abs(perturbed_preds - baseline_preds[start:start+batch_size]).sum()
                    batches_processed += 1

                    # Memory optimization
                    if batches_processed % 10 == 0:
                        gc.collect()
                        tf.keras.backend.clear_session()

                mean_diff = total_diff / n_samples
                delta_sens += mean_diff
                print(f"\n  ✓ Direction complete: {time.time()-dir_start:.1f}s | Δ={mean_diff:.4f}")

            # Update sensitivity
            feature_sens += delta_sens / 2  # Average directions
            print(f"  🕒 Perturbation time: {time.time()-delta_start:.1f}s")
        
        sensitivities[feat_idx] = feature_sens / len(deltas)
        print(f"\n📊 Feature {feat_idx+1} complete: {sensitivities[feat_idx]:.4f}")
        print(f"⏱️ Feature time: {time.time()-feature_start:.1f}s")
        print(f"🔮 Estimated remaining: {(time.time()-start_time)*(n_features-feat_idx-1)/(feat_idx+1):.1f}s")
        print("──────────────────────────────────────────────────")
    
    total_time = time.time() - start_time
    print(f"\n✅ Analysis completed in {total_time//3600:.0f}h {total_time%3600//60:.0f}m {total_time%60:.0f}s!")
    print(f"• Peak RAM usage: {psutil.virtual_memory().percent}%")
    return sensitivities


# Run the improved sensitivity analysis
print("\nPerforming improved feature sensitivity analysis...")
sensitivities = improved_sensitivity_analysis(discriminator, X_scaled, baseline_preds, batch_size,
                                              perturbation_factors=[0.5, 1.0, 2.0, 5.0, 10])

# Display individual feature sensitivity scores
print("\nFeature Sensitivity Scores:")
for feature, sens in zip(feature_names, sensitivities):
    print(f"Feature: {feature}, Sensitivity: {sens:.4f}")




Performing improved feature sensitivity analysis...
[1.64093139e-05 1.52594876e-05 5.00000000e-01 6.23398645e-06
 1.56985871e-03 1.21951220e-02 2.95945546e-04 6.84931507e-03
 6.88705234e-04 6.89655172e-04 2.74499039e-04 3.26690624e-04
 3.70370370e-02 1.11111111e-01 3.55871886e-03 3.74531835e-03
 1.42275440e-06 2.36233493e-06 1.56194844e-06 4.43212511e-07
 1.12415127e-05 1.96850394e-03 6.26036874e-06 1.56424022e-06
 4.43489267e-07 1.12851533e-05 2.25733634e-03 3.06748466e-04
 2.97796307e-04 2.91460216e-04 3.52360817e-04 2.17391304e-02
 1.00000000e+00 1.00000000e-03 1.00000000e-03 1.00000000e-03
 8.37380673e-05 7.63358779e-03 2.37835318e-06 1.86636805e-04
 6.89655172e-04 6.89179876e-04 2.28623685e-04 2.98775022e-04
 2.72034820e-04 1.00000000e-03 1.00000000e+00 1.00000000e+00
 1.00000000e-03 1.00000000e+00 1.00000000e+00 1.00000000e+00
 1.00000000e-03 1.66666667e-01 1.94969780e-04 3.04971028e-04
 3.87596899e-03 8.37380673e-05 1.00000000e-03 1.00000000e-03
 1.00000000e-03 1.00000000e-03 1

NameError: name 'sensitivities' is not defined

In [17]:
# Rank features by sensitivity
sorted_idx = np.argsort(-sensitivities)
print("\nRanking of Features by Sensitivity:")
for i, idx in enumerate(sorted_idx): 
    print(f"{i+1}. {feature_names[idx]}: {sensitivities[idx]:.4f}")


Ranking of Features by Sensitivity:
1. Inbound: 0.0579
2. Protocol: 0.0194
3. URG Flag Count: 0.0142
4. Down/Up Ratio: 0.0094
5. ACK Flag Count: 0.0054
6. Fwd PSH Flags: 0.0043
7. RST Flag Count: 0.0027
8. Bwd Packet Length Min: 0.0022
9. CWE Flag Count: 0.0022
10. SYN Flag Count: 0.0021
11. Bwd IAT Min: 0.0014
12. Subflow Bwd Bytes: 0.0013
13. Bwd Packet Length Max: 0.0013
14. Init_Win_bytes_backward: 0.0010
15. Bwd Header Length: 0.0006
16. act_data_pkt_fwd: 0.0006
17. Total Length of Bwd Packets: 0.0005
18. Active Std: 0.0005
19. Active Min: 0.0004
20. Init_Win_bytes_forward: 0.0004
21. Total Backward Packets: 0.0004
22. Subflow Bwd Packets: 0.0003
23. Fwd Packet Length Min: 0.0002
24. Fwd Packet Length Max: 0.0002
25. Avg Bwd Segment Size: 0.0002
26. Bwd Packet Length Std: 0.0001
27. Active Mean: 0.0001
28. Fwd IAT Min: 0.0001
29. Max Packet Length: 0.0001
30. Active Max: 0.0001
31. Subflow Fwd Bytes: 0.0001
32. Subflow Fwd Packets: 0.0001
33. Bwd Packet Length Mean: 0.0001
34. Av

In [18]:
# Save results to file
results_df = pd.DataFrame({
    'Feature': feature_names,
    'Sensitivity_Score': sensitivities
}).sort_values('Sensitivity_Score', ascending=False)

results_df.to_csv('feature_sensitivity_results.csv', index=False)
print("\nResults saved to 'feature_sensitivity_results2.csv'")


Results saved to 'feature_sensitivity_results2.csv'


In [19]:
import itertools 
# Step 5. Feature Combination Analysis
def feature_pair_sensitivity(model, data, baseline_preds, individual_sensitivities, batch_size, top_n,  delta=0.1, safety_factor=0.5):
    """Mission-critical interaction analysis with intelligent memory handling"""
    n_samples, _ = data.shape
    data = data.astype(np.float32)
    pair_results = []
    start_time = time.time()
    top_indices = np.argsort(-individual_sensitivities)[:top_n]
    total_pairs = len(list(itertools.combinations(top_indices, 2)))
    
    batches_per_pair = (n_samples + batch_size - 1) // batch_size
    
    print(f"\n🚀 Launching pair analysis with:")
    print(f"• Total pairs: {total_pairs}")
    print(f"• Samples per pair: {n_samples:,}")
    print(f"• Batches per pair: {batches_per_pair}")
    print(f"• Estimated total batches: {total_pairs * batches_per_pair:,}")
    print("──────────────────────────────────────────────────")

    for pair_idx, (i, j) in enumerate(itertools.combinations(top_indices, 2)):
        pair_start = time.time()
        feat1, feat2 = feature_names[i], feature_names[j]
        total_diff = 0.0
        
        # Pair header with progress tracking
        print(f"\n🔗 Pair {pair_idx+1}/{total_pairs} [{'▋'*(20*(pair_idx+1)//total_pairs)}{' '*(20-20*(pair_idx+1)//total_pairs)}]")
        print(f"• Features: {feat1} & {feat2}")
        print(f"• Elapsed: {time.time()-start_time:.1f}s")
        print(f"• Current RAM: {psutil.virtual_memory().percent}%")
        print("──────────────────────────────────────────────────")

        for batch_idx in range(0, n_samples, batch_size):
            batch_start = time.time()
            end = min(batch_idx + batch_size, n_samples)
            batch_data = data[batch_idx:end].copy()
            
            # Apply dual perturbation
            batch_data[:, [i, j]] = np.clip(batch_data[:, [i, j]] + delta, 0, 1)
            
            # Compute impact
            perturbed_preds = model(batch_data).numpy()
            total_diff += np.abs(perturbed_preds - baseline_preds[batch_idx:end]).sum()
            
            # Progress tracking every 5% of batches or 10 batches minimum
            current_batch = (batch_idx // batch_size) + 1
            if current_batch % max(10, batches_per_pair//20) == 0 or current_batch == batches_per_pair:
                elapsed = time.time() - pair_start
                progress = (current_batch/batches_per_pair)*100
                mem = psutil.virtual_memory()
                print(
                    f"\r▏{'█' * int(progress//5)}{' ' * (20 - int(progress//5))}▏ "
                    f"{progress:.1f}% | "
                    f"Batch {current_batch}/{batches_per_pair} | "
                    f"RAM: {mem.percent}% | "
                    f"Time: {elapsed:.1f}s", end="", flush=True
                )
            
            # Memory maintenance every 10 batches
            if current_batch % 10 == 0:
                gc.collect()
                tf.keras.backend.clear_session()
                print(f"\n♻️ Memory cleaned | RAM: {psutil.virtual_memory().percent}%")

        # Calculate metrics
        combined_sens = total_diff / n_samples
        interaction = combined_sens - (individual_sensitivities[i] + individual_sensitivities[j])
        
        pair_results.append({
            'features': (feat1, feat2),
            'combined_impact': combined_sens,
            'synergy_score': interaction,
            'individual_sum': individual_sensitivities[i] + individual_sensitivities[j]
        })
        
        # Pair summary
        pair_time = time.time() - pair_start
        print(f"\n\n✅ Pair complete:")
        print(f"  ⚡ Combined impact: {combined_sens:.4f}")
        print(f"  💥 Synergy score: {interaction:.4f}")
        print(f"  ⏱️ Processing time: {pair_time:.1f}s")
        print(f"  🔮 Estimated remaining: {(time.time()-start_time)*(total_pairs-pair_idx-1)/(pair_idx+1):.1f}s")
        print("──────────────────────────────────────────────────")

    # Final summary
    total_time = time.time() - start_time
    print(f"\n🎉 Analysis completed in {total_time//3600:.0f}h {total_time%3600//60:.0f}m {total_time%60:.0f}s!")
    print(f"• Total pairs processed: {total_pairs}")
    print(f"• Peak RAM usage: {psutil.virtual_memory().percent}%")
    print(f"• Final batch size: {batch_size}")
    
    return sorted(pair_results, key=lambda x: x['synergy_score'], reverse=True)


# Analyze feature combinations
print("\nAnalyzing feature pair interactions...")
pair_results = feature_pair_sensitivity(discriminator, X_scaled, baseline_preds, sensitivities, batch_size, top_n=20)



Analyzing feature pair interactions...

🚀 Launching pair analysis with:
• Total pairs: 190
• Samples per pair: 27,974,480
• Batches per pair: 14
• Estimated total batches: 2,660
──────────────────────────────────────────────────

🔗 Pair 1/190 [                    ]
• Features: Inbound & Protocol
• Elapsed: 0.0s
• Current RAM: 19.9%
──────────────────────────────────────────────────
▏██████████████      ▏ 71.4% | Batch 10/14 | RAM: 20.1% | Time: 7.0s
♻️ Memory cleaned | RAM: 20.1%
▏████████████████████▏ 100.0% | Batch 14/14 | RAM: 20.0% | Time: 10.0s

✅ Pair complete:
  ⚡ Combined impact: 0.0000
  💥 Synergy score: -0.0772
  ⏱️ Processing time: 10.0s
  🔮 Estimated remaining: 1889.2s
──────────────────────────────────────────────────

🔗 Pair 2/190 [                    ]
• Features: Inbound & URG Flag Count
• Elapsed: 10.0s
• Current RAM: 20.0%
──────────────────────────────────────────────────
▏██████████████      ▏ 71.4% | Batch 10/14 | RAM: 20.1% | Time: 7.1s
♻️ Memory cleaned | RAM: 2

In [20]:
print("\nTop Feature Pairs with Synergistic Effects:")
for i, pair in enumerate(pair_results): # Show top 10 pairs if pair_results already holds the top pairs
    feat1, feat2 = pair['features']
    print(f"{i+1}. {feat1} + {feat2}: Combined Impact={pair['combined_impact']:.4f}, Synergy Score={pair['synergy_score']:.4f}, Individual Sum={pair['individual_sum']:.4f}")


Top Feature Pairs with Synergistic Effects:
1. Subflow Bwd Bytes + Active Min: Combined Impact=0.0199, Synergy Score=0.0182, Individual Sum=0.0017
2. Subflow Bwd Bytes + Active Std: Combined Impact=0.0165, Synergy Score=0.0147, Individual Sum=0.0018
3. Bwd Packet Length Min + Subflow Bwd Bytes: Combined Impact=0.0176, Synergy Score=0.0141, Individual Sum=0.0035
4. Active Std + Active Min: Combined Impact=0.0149, Synergy Score=0.0139, Individual Sum=0.0009
5. Bwd Packet Length Min + Active Min: Combined Impact=0.0163, Synergy Score=0.0137, Individual Sum=0.0026
6. Bwd IAT Min + Subflow Bwd Bytes: Combined Impact=0.0153, Synergy Score=0.0126, Individual Sum=0.0026
7. Bwd IAT Min + Active Min: Combined Impact=0.0136, Synergy Score=0.0119, Individual Sum=0.0018
8. Bwd Header Length + act_data_pkt_fwd: Combined Impact=0.0117, Synergy Score=0.0105, Individual Sum=0.0012
9. act_data_pkt_fwd + Total Length of Bwd Packets: Combined Impact=0.0113, Synergy Score=0.0102, Individual Sum=0.0011
10.

In [21]:
# Convert results to a DataFrame
results_df = pd.DataFrame({
    'Feature 1': [pair['features'][0] for pair in pair_results],
    'Feature 2': [pair['features'][1] for pair in pair_results],
    'Combined Impact': [pair['combined_impact'] for pair in pair_results],
    'Synergy Score': [pair['synergy_score'] for pair in pair_results],
    'Individual Sum': [pair['individual_sum'] for pair in pair_results]
}).sort_values('Synergy Score', ascending=False)

# Save results to CSV
results_df.to_csv('feature_pair_interactions.csv', index=False)
print("\nResults saved to 'feature_pair_interactions.csv'")



Results saved to 'feature_pair_interactions.csv'
